In [ ]:
from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.data_downloader import download_data
from common.data import get_dataloaders, preprocess_spectra, normalize, denormalize, SpectraDataset
from common.models import SpectraUncertaintyCNN
from common.losses import gaussian_nll_loss
from common.training import train_model, load_checkpoint

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, r2_score

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks 
%matplotlib inline  
import matplotlib.pyplot as plt   # THEN import pyplot

In [ ]:
# download data for this task, if needed and get path 
data_path = download_data("task_01") # Task 01 data is used for task 02 as well, so we can reuse it here
# get and if needed create figure directory for this task
fig_dir = get_figure_dir("task_01")

In [ ]:
spectra = np.load(f"{data_path}/spectra.npy")
spectra_length = spectra.shape[1]

# labels: mass, age, l_bol, dist, t_eff, log_g, fe_h, SNR
labelNames = ["mass", "age", "l_bol", "dist", "t_eff", "log_g", "fe_h", "SNR"]
labels = np.load(f"{data_path}/labels.npy")

# We only use the three labels: t_eff, log_g, fe_h
labelNames = labelNames[-4:-1]
labels = labels[:, -4:-1]
n_labels = labels.shape[1]


# preprocess spectra
spectra = preprocess_spectra(spectra)

# normalize spectra
spectra_norm, spectra_mean, spectra_std = normalize(spectra)

# normalize labels column-wise
labels_norm, label_mean, label_std = normalize(
    labels,
    axis=0,
)

# create dataset
dataset = SpectraDataset(
    spectra_norm,
    labels_norm,
)

# create dataloaders
train_loader, val_loader, test_loader = get_dataloaders(
    dataset,
    batch_size=64,
)

In [ ]:
# Check device availability
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
model = SpectraUncertaintyCNN(n_labels=3).to(device)
model.eval()

x, y = next(iter(train_loader))
x = x.to(device)
y = y.to(device)

with torch.no_grad():
    outputs = model(x)
    mu, log_var = outputs
    loss = gaussian_nll_loss(outputs, y)

print(f"x:       {x.shape}")
print(f"y:       {y.shape}")
print(f"mu:      {mu.shape}")
print(f"log_var: {log_var.shape}")
print(f"loss:    {loss.item():.4f}")
print(f"log_var range: [{log_var.min().item():.3f}, {log_var.max().item():.3f}]")

assert x.ndim == 3 and x.shape[1] == 1
assert mu.shape == y.shape
assert log_var.shape == y.shape
assert loss.ndim == 0
assert torch.isfinite(x).all()
assert torch.isfinite(y).all()
assert torch.isfinite(mu).all()
assert torch.isfinite(log_var).all()
assert torch.isfinite(loss)

In [ ]:
model_configs = [
    {
        "run_name": "k3_c16-32-64_drop0.1_bn",
        "channels": (16, 32, 64),
        "kernel_size": 3,
        "convs_per_block": 2,
        "dropout": 0.1,
        "adaptive_pool_length": 64,
        "hidden_dim": 128,
        "use_batchnorm": True,
    },
    {
        "run_name": "k3_c16-32-64_drop0.0_bn",
        "channels": (16, 32, 64),
        "kernel_size": 3,
        "convs_per_block": 2,
        "dropout": 0.0,
        "adaptive_pool_length": 64,
        "hidden_dim": 128,
        "use_batchnorm": True,
    },
    {
        "run_name": "k5_c16-32-64_drop0.1_bn",
        "channels": (16, 32, 64),
        "kernel_size": 5,
        "convs_per_block": 2,
        "dropout": 0.1,
        "adaptive_pool_length": 64,
        "hidden_dim": 128,
        "use_batchnorm": True,
    },
    {
        "run_name": "k3_c32-64-128_drop0.1_bn",
        "channels": (32, 64, 128),
        "kernel_size": 3,
        "convs_per_block": 2,
        "dropout": 0.1,
        "adaptive_pool_length": 64,
        "hidden_dim": 128,
        "use_batchnorm": True,
    },
    {
        "run_name": "k3_c16-32-64-128_drop0.1_pool32_bn",
        "channels": (16, 32, 64, 128),
        "kernel_size": 3,
        "convs_per_block": 2,
        "dropout": 0.1,
        "adaptive_pool_length": 32,
        "hidden_dim": 128,
        "use_batchnorm": True,
    },
    {
        "run_name": "k9_c16-32-64_1conv_drop0.2_no_bn",
        "channels": (16, 32, 64),
        "kernel_size": 9,
        "convs_per_block": 1,
        "dropout": 0.2,
        "adaptive_pool_length": 64,
        "hidden_dim": 128,
        "use_batchnorm": False,
    },
]

In [ ]:
results = []

num_epochs = 100
patience = 10
skip_training = True

for config in model_configs:
    run_name = config["run_name"]
    print(f"\nRunning experiment: {run_name}")

    model = SpectraUncertaintyCNN(
        n_labels=n_labels,
        channels=config["channels"],
        kernel_size=config["kernel_size"],
        convs_per_block=config["convs_per_block"],
        dropout=config["dropout"],
        adaptive_pool_length=config["adaptive_pool_length"],
        hidden_dim=config["hidden_dim"],
        use_batchnorm=config["use_batchnorm"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
    )

    checkpoint_path = f"../models/{run_name}_best.pth"

    train_losses, val_losses = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=gaussian_nll_loss,
        device=device,
        num_epochs=num_epochs,
        checkpoint_path=checkpoint_path,
        patience=patience,
        skip_training=skip_training,
        extra_checkpoint_info={"model_config": config},
        verbose=True,
    )

    results.append({
        "run_name": run_name,
        "best_val_loss": min(val_losses),
        "best_epoch": int(np.argmin(val_losses)) + 1,
        "checkpoint_path": checkpoint_path,
    })
    print(f"Finished {run_name}: best_val_loss={min(val_losses):.4f}")

In [ ]:
results_df = pd.DataFrame(results).sort_values("best_val_loss")
results_df

In [ ]:
best_run = results_df.iloc[0]

best_model_path = best_run["checkpoint_path"]
best_run_name = best_run["run_name"]

print("Best run:", best_run_name)
print("Checkpoint path:", best_model_path)

checkpoint = load_checkpoint(
    best_model_path,
    device=device,
)

print("Best validation loss:", checkpoint["best_val_loss"])
print("Best epoch:", checkpoint["epoch"])

train_losses = checkpoint["train_losses"]
val_losses = checkpoint["val_losses"]

In [ ]:
# Recreate model architecture from saved config
config = checkpoint["model_config"]

model = SpectraUncertaintyCNN(
    n_labels=n_labels,
    channels=config["channels"],
    kernel_size=config["kernel_size"],
    convs_per_block=config["convs_per_block"],
    dropout=config["dropout"],
    adaptive_pool_length=config["adaptive_pool_length"],
    hidden_dim=config["hidden_dim"],
    use_batchnorm=config["use_batchnorm"],
).to(device)

# Load learned weights
model.load_state_dict(checkpoint["model_state_dict"])

# Switch to evaluation mode
model.eval()

# Evaluate on test set
test_loss = 0.0
mu_list = []
log_var_list = []
true_labels_list = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        mu, log_var = model(batch_x)

        mu_list.append(mu.cpu())
        log_var_list.append(log_var.cpu())
        true_labels_list.append(batch_y.cpu())

        loss = gaussian_nll_loss((mu, log_var), batch_y)
        test_loss += loss.item()

# Concatenate predictions, uncertainties, and true labels
preds = torch.cat(mu_list, dim=0)
log_vars = torch.cat(log_var_list, dim=0)
truths = torch.cat(true_labels_list, dim=0)

# Convert log variance to standard deviation
sigmas = torch.exp(0.5 * log_vars)

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# Train and validation loss curves
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(train_losses, label="Train Loss (Train Mode)", color="C0")
ax.plot(val_losses, label="Validation Loss", color="C1")

ax.set_xlabel("Epoch")
ax.set_ylabel("Gaussian NLL Loss")
ax.set_title("Training and Validation Loss Curves")

ax.axhline(
    test_loss,
    color="C2",
    linestyle="--",
    label=f"Test Loss: {test_loss:.4f}",
)

ax.axvline(
    checkpoint["epoch"] - 1, # checkpoint epoch is 1-based, but x-axis is 0-based
    color="black",
    linestyle=":",
    label=f"Best Checkpoint Epoch: {checkpoint['epoch']}",
)

ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/loss_curves.pdf")
None

In [ ]:
# Convert predictions and true labels back to original scale
predictions_orig = denormalize(preds.numpy(), label_mean, label_std)
true_labels_orig = denormalize(truths.numpy(), label_mean, label_std)

# Convert predicted uncertainties back to original scale
sigmas_orig = sigmas.numpy() * label_std

latex_names = [
    r"$T_{\mathrm{eff}}$",
    r"$\log g$",
    r"$[\mathrm{Fe}/\mathrm{H}]$",
]

fig, axes = plt.subplots(1, 3, figsize=(8, 4))

for i in range(n_labels):
    ax = axes[i]

    x_true = true_labels_orig[:, i]
    y_pred = predictions_orig[:, i]
    y_err = sigmas_orig[:, i]

    ax.errorbar(
        x_true,
        y_pred,
        yerr=y_err,
        fmt="o",
        ms=4,
        alpha=0.5,
        color="C0",
        ecolor="C0",
        elinewidth=0.8,
        capsize=1.5,
        mec="k",
        mew=0.3,
        linestyle="none",
        label="Test Set",
    )

    lo = min(x_true.min(), y_pred.min())
    hi = max(x_true.max(), y_pred.max())

    ax.plot(
        [lo - 0.01 * (hi - lo), hi + 0.01 * (hi - lo)],
        [lo, hi],
        color="red",
        linestyle="--",
        linewidth=1.2,
        label="Ideal fit",
        zorder=3,
    )

    ax.set_xlabel(f"True {latex_names[i]}")
    ax.set_ylabel(f"Predicted {latex_names[i]}")
    ax.set_title(f"{latex_names[i]}")

    mae = mean_absolute_error(x_true, y_pred)
    r2 = r2_score(x_true, y_pred)

    metrics_text = f"MAE: {mae:.4f}\n$R^2$: {r2:.4f}"
    ax.text(
        0.05,
        0.95,
        metrics_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round", alpha=0.1),
    )

    ax.legend(loc="lower right")

fig.suptitle("Model Predictions vs True Labels on Test Set", fontsize=15)
fig.tight_layout()
fig.savefig(f"{fig_dir}/predictions_vs_true_with_uncertainty.pdf")
None

In [ ]:
# Pulls in normalized label units
pulls = (truths.numpy() - preds.numpy()) / sigmas.numpy()

print(pulls.shape)  # (N_test, 3)

label_names = ["t_eff", "log_g", "fe_h"]

for name, pull_i in zip(label_names, pulls.T):
    print(name)
    print(f"  pull mean: {pull_i.mean():.3f}")
    print(f"  pull std:  {pull_i.std():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))

for i in range(n_labels):
    ax = axes[i]

    pull_i = pulls[:, i]

    pull_mean = pull_i.mean()
    pull_std = pull_i.std()

    ax.hist(
        pull_i,
        bins=40,
        density=True,
        alpha=0.7,
        color="C0",
        edgecolor="k",
        label="Pull distribution",
    )

    x = np.linspace(-5, 5, 300)
    standard_normal = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)

    ax.plot(
        x,
        standard_normal,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=r"$\mathcal{N}(0, 1)$",
    )

    ax.axvline(0, color="black", linestyle=":", linewidth=1)

    ax.set_xlabel(r"Pull $(y - \mu) / \sigma$")
    ax.set_ylabel("Density")
    ax.set_title(latex_names[i])

    metrics_text = f"Mean: {pull_mean:.3f}\nStd: {pull_std:.3f}"
    ax.text(
        0.05,
        0.7,
        metrics_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round", alpha=0.1),
    )

    ax.legend(loc="upper left")

fig.suptitle("Pull Distributions on Test Set", fontsize=15)
fig.tight_layout()
fig.savefig(f"{fig_dir}/pull_distributions.pdf")
None